# 05 — Exercises

Write the code yourself. Solutions in `dags/exercises/`.

---
## Exercise 1 — Basic DAG with dependencies

Write a DAG called `asin_daily_pipeline` that:
- Runs daily at 7am UTC
- Has 4 tasks: `extract` → `validate` → `transform` → `load`
- `extract` prints `"extracting for {{ ds }}"`
- `validate` prints row count (hardcode 50000)
- `transform` prints `"transforming"`
- `load` prints `"loading to redshift"`
- `retries=2`, `retry_delay=5min` on all tasks

In [ ]:
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

# TODO: define default_args

# TODO: define extract(), validate(), transform(), load() functions

# TODO: create DAG with correct schedule and settings

# TODO: create 4 PythonOperator tasks

# TODO: set dependencies extract >> validate >> transform >> load

---
## Exercise 2 — XCom + BranchPythonOperator

Write a DAG called `xcom_pipeline` where:
- `count_rows` returns 75000 (auto-pushed to XCom)
- `check_threshold` pulls that count — if > 50000 returns `"process_full"`, else `"process_sample"`
- Both branches end at a `complete` task
- `complete` must run regardless of which branch ran

In [ ]:
from airflow.operators.python import BranchPythonOperator
from airflow.operators.empty import EmptyOperator

# TODO: count_rows function → returns 75000

# TODO: check_threshold function → pulls XCom, returns branch task_id

# TODO: process_full function

# TODO: process_sample function

# TODO: build DAG with correct trigger_rule on complete task

# TODO: set dependencies

---
## Exercise 3 — Sensor + downstream

Write a DAG called `sensor_pipeline` that:
- Waits for `/tmp/{{ ds_nodash }}/data.parquet` (FileSensor, local dev)
- Uses `reschedule` mode, pokes every 2 min, times out after 4 hours
- On timeout → mark SKIPPED (not FAILED)
- On file arrival → runs a `process` task

In [ ]:
from airflow.sensors.filesystem import FileSensor

# TODO: define process function

# TODO: build DAG with FileSensor and process task

# What trigger_rule does process need if soft_fail=True on the sensor?
# Think about it before checking the solution.

---
## Exercise 4 — Dynamic regional DAG

Write a DAG called `regional_glue_pipeline` that:
- Runs a task for each of `["na", "eu", "fe"]`
- All 3 run in parallel (use TaskGroup)
- Each prints: `"running asin-processing-{region} for {ds}"`
- After all 3 complete, run a single `aggregate` task

In [ ]:
from airflow.utils.task_group import TaskGroup

REGIONS = ["na", "eu", "fe"]

# TODO: process_region function (takes region as arg)

# TODO: aggregate function

# TODO: build DAG with TaskGroup for regional tasks and aggregate at end

---
## Exercise 5 — Rewrite Exercise 2 using TaskFlow API

Same logic as Exercise 2 (`count_rows` → branch → `process_full` or `process_sample` → `complete`)
but using `@dag` and `@task` decorators. XCom must be implicit.

In [ ]:
from airflow.decorators import dag, task

# TODO: @dag definition

# TODO: @task count_rows → returns int

# TODO: @task.branch check_threshold(count: int) → returns task_id string

# TODO: @task process_full

# TODO: @task process_sample

# TODO: EmptyOperator complete with correct trigger_rule

# TODO: wire the tasks and call the dag function

---
## Interview Questions — answer in the cells below

**Q1:** You have a DAG running hourly. One run is stuck at a Glue task for 3 hours. What happens to subsequent runs?

*Your answer:*

**Q2:** A task needs data from the previous day's DAG run. How do you access it?

*Your answer:*

**Q3:** Your MWAA pipeline waits for S3 files that sometimes arrive 2 hours late. How do you handle it?

*Your answer:*

**Q4:** A downstream task should run whether upstream succeeded OR failed (cleanup). What trigger_rule?

*Your answer:*

**Q5:** You want to pass a 500MB DataFrame between tasks. How?

*Your answer:*

**Q6:** What's the difference between clearing a task and marking it as success?

*Your answer:*

**Q7:** Your pipeline SLA is 8am but it ran at 8:30am. How do you investigate in MWAA?

*Your answer:*